# Module 01 — NumPy Foundations

*Measuring 150 flowers, the fast way.*

In 1936, botanist Edgar Anderson measured the sepals and petals of 150 iris flowers from 3 species. We'll use his data to learn **NumPy** — the array library underneath all of scientific Python.

Think of the data as a matrix, just like a gene expression matrix:

- **rows** = individuals (flowers ↔ samples)
- **columns** = measurements (petal length ↔ genes)

**How to use this notebook:** run each cell in order with `Shift+Enter`, and *predict the output before you run it*. Every code line that isn't obvious has a comment (the text after `#`) explaining it — Python ignores everything after a `#`, it's purely for humans.

In [3]:
# 'import ... as ...' loads a library and gives it a short nickname,
# so we can type 'np.' instead of 'numpy.' everywhere.
import numpy as np                # numerical arrays -- the star of this module
import pandas as pd               # tables (brief preview here; the star of Module 02)
import matplotlib.pyplot as plt   # drawing plots
import seaborn as sns             # nicer-looking plots, built on top of matplotlib

# Set one consistent, colorblind-safe style for every plot in this course.
sns.set_theme(style="whitegrid", palette="colorblind")

## 1. Load the data from HuggingFace

[HuggingFace](https://huggingface.co/datasets) hosts thousands of datasets. `load_dataset` downloads one for us (only the first time — after that it loads instantly from a local cache).

In [4]:
from datasets import load_dataset   # the HuggingFace dataset downloader

# Download the iris dataset (its address is "scikit-learn/iris").
iris = load_dataset("scikit-learn/iris", split="train")

# Convert it into a table and show the first 5 rows.
# (These 'DataFrame' tables get their own module next -- for now, just look.)
df = iris.to_pandas()
df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 2. From table to array

For this module we pull the numbers out into a NumPy **array** — a pure grid of numbers.

- `shape` tells you the dimensions: `(150, 4)` = 150 flowers × 4 measurements
- `dtype` (data type) tells you the type of *every* element — unlike a Python list, an array holds one single type, and that's part of what makes it fast

In [ ]:
# The four measurement columns we want, listed by name.
feature_names = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]

# df[...] picks those columns; .to_numpy() turns them into a plain NumPy array.
X = df[feature_names].to_numpy()     # the 150 x 4 measurement matrix
species = df["Species"].to_numpy()   # one species label per flower (150 labels)

print("shape:", X.shape)   # (rows, columns) = (flowers, measurements)
print("dtype:", X.dtype)   # 'float64' means decimal numbers
print("first flower:", X[0], "->", species[0])   # X[0] = row 0 = the first flower

## 3. Why bother? Vectorization

NumPy does math on **the whole array at once**, in fast compiled code, instead of visiting each value in a slow Python loop. That's called **vectorization**.

On 150 flowers you won't feel the difference — so let's simulate something bigger: expression levels for the ~20,000 genes in a human genome, and time both approaches.

In [ ]:
# Make a random-number generator. 'seed=42' means it starts from a fixed point,
# so everyone gets the SAME "random" numbers on every run -- reproducibility.
rng = np.random.default_rng(seed=42)

# Draw 20,000 values from a bell curve centered at 100 with spread 20:
# one fake expression level per gene. (20_000 is just 20000 -- the underscore
# is only there to make the number easier to read.)
expression = rng.normal(loc=100, scale=20, size=20_000)

# %timeit is a notebook 'magic command': it runs the line many times and
# reports the average speed. (It works in notebooks, not in .py files.)

# The Python way -- sum() visits all 20,000 values one by one:
%timeit -n 10 sum(expression) / len(expression)

# The NumPy way -- one vectorized call handles all values at once:
%timeit -n 10 expression.mean()

NumPy is typically **hundreds of times faster** (compare the units: µs = microseconds are 1000× smaller than ms = milliseconds). On a real single-cell RNA-seq matrix — thousands of cells × 20k genes — this is the difference between seconds and hours.

> **The core habit:** don't loop over data — describe the operation on the whole array.

## 4. Indexing and slicing

Square brackets pick values out of an array, and counting **starts at 0** (so "flower 0" is the first flower). With two dimensions it's `X[row, column]`.

In [ ]:
print("flower 0, all 4 measurements:  ", X[0])        # the whole first row
print("petal length of flower 0:       ", X[0, 2])     # row 0, column 2 (3rd column!)
print("petal lengths of first 5 flowers:", X[:5, 2])   # ':5' means rows 0 through 4

# A ':' by itself means 'ALL of them'. So X[:, 2] = every row, column 2.
petal_length = X[:, 2]   # all 150 petal lengths in one array
petal_width  = X[:, 3]   # all 150 petal widths
print("all 150 petal lengths, shape:", petal_length.shape)

## 5. Boolean masking — filtering, the array way

Comparing an array to a value asks the question of **every element at once**, and gives back an array of `True`/`False` answers — a **mask**. Using that mask as an index keeps only the `True` rows.

This is *the* filtering idiom of scientific Python — you'll use it in pandas, scikit-learn, and scanpy.

In [ ]:
# One comparison, 150 answers: is each flower a setosa?
is_setosa = species == "Iris-setosa"
print("mask (first 5):", is_setosa[:5])

# .sum() on True/False values counts the Trues (True counts as 1, False as 0).
print("number of setosa flowers:", is_setosa.sum())

# Index with the mask -> keep only the rows where the mask says True.
setosa_petals = petal_length[is_setosa]
print("mean setosa petal length:", setosa_petals.mean().round(2), "cm")

In [ ]:
# np.unique() lists each different species name once.
for name in np.unique(species):
    mask = species == name                        # True for flowers of THIS species
    mean_length = petal_length[mask].mean()       # average over just those flowers

    # An f-string (the f before the quote) drops values into text at the {braces};
    # ':.2f' inside a brace means 'show 2 decimal places'.
    print(f"{name}: mean petal length {mean_length:.2f} cm")

Setosa petals are ~1.5 cm, virginica ~5.5 cm — nearly 4× longer. Measurements really do carry species identity. Let's *see* it.

## 6. First plot: distributions by species

A histogram shows the **distribution** of a measurement — the range of values is split into bins, and the bar height counts how many flowers land in each bin.

In [ ]:
# A dictionary pairs each species name with a fixed color, so the SAME species
# gets the SAME color in every plot of this course.
palette = sns.color_palette("colorblind")   # a list of colorblind-safe colors
SPECIES_COLORS = {
    "Iris-setosa": palette[0],       # blue
    "Iris-versicolor": palette[1],   # orange
    "Iris-virginica": palette[2],    # green
}

# Create an empty figure containing one drawing area ('ax' = the axes).
fig, ax = plt.subplots(figsize=(8, 4.5))    # figsize = (width, height) in inches

# .items() loops over the dictionary's (name, color) pairs, one species at a time.
for name, color in SPECIES_COLORS.items():
    ax.hist(petal_length[species == name],    # only this species' petal lengths
            bins=12,                          # split the range into 12 bars
            alpha=0.8,                        # slightly see-through, so overlaps show
            color=color,
            label=name.replace("Iris-", ""))  # legend text without the 'Iris-' prefix

ax.set_xlabel("Petal length (cm)")            # always label your axes!
ax.set_ylabel("Number of flowers")
ax.set_title("Petal length almost perfectly separates the three species")
ax.legend(title="Species")                    # the color key
plt.show()                                    # display the finished figure

Setosa is completely separate; versicolor and virginica overlap a little. One measurement gets you most of the way to a species ID.

## 7. Vectorized math — new measurements from old

Arithmetic between two arrays happens **element by element**: flower 0 with flower 0, flower 1 with flower 1, and so on. One expression, no loop.

In [ ]:
# 150 multiplications at once: each flower's length x its own width.
petal_area = petal_length * petal_width
print("first 5 petal areas (cm^2):", petal_area[:5].round(2))

# .argmax() returns the POSITION of the largest value (not the value itself).
champion = petal_area.argmax()

print()   # just prints a blank line
print(f"Biggest petals: flower #{champion} ({species[champion]}), "
      f"{petal_length[champion]} x {petal_width[champion]} cm")

## 8. Broadcasting and z-scores

Can you subtract a row of 4 numbers from a 150×4 matrix? In NumPy, yes: it automatically "stretches" the small array across the big one. That's called **broadcasting**.

We'll use it to compute the **z-score**: *how many standard deviations is this flower from the average?* (Standard deviation = the typical distance of values from their mean.) This is the same standardization applied to gene expression data before clustering, and scikit-learn will automate it later (`StandardScaler`).

One new idea: `axis=0` means "compute down the columns" (one answer per measurement); `axis=1` would go across rows (one answer per flower).

In [ ]:
col_means = X.mean(axis=0)   # 4 numbers: the average of each measurement column
col_stds  = X.std(axis=0)    # 4 numbers: the spread of each measurement column
print("column means:", col_means.round(2))

# Broadcasting in action: (150 x 4) minus (4,) works -- the 4 means are
# subtracted from every row, then each column is divided by its own spread.
X_z = (X - col_means) / col_stds

# Sanity check: after z-scoring, every column should average ~0 with spread ~1.
print("z-scored means (~0):", X_z.mean(axis=0).round(2))
print("z-scored stds  (~1):", X_z.std(axis=0).round(2))
print("flower 0 in z-scores:", X_z[0].round(2))

## 9. Two measurements at once: the scatter plot

If one measurement almost separates species, two should do even better. Each dot is one flower; color = species.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))

for name, color in SPECIES_COLORS.items():
    mask = species == name             # pick out this species' flowers
    ax.scatter(petal_length[mask],     # x position of each dot
               petal_width[mask],      # y position of each dot
               s=45,                   # dot size
               alpha=0.85,             # slight transparency
               color=color,
               edgecolor="white",      # white rim keeps overlapping dots readable
               linewidth=0.6,
               label=name.replace("Iris-", ""))

ax.set_xlabel("Petal length (cm)")
ax.set_ylabel("Petal width (cm)")
ax.set_title("Two petal measurements are enough to tell the species apart")
ax.legend(title="Species")
plt.show()

Three clean clouds. When a machine learning model "classifies" iris species, all it's really doing is drawing boundaries between these clouds — which you can now basically do by eye.

## Recap

| Idea | Code |
|---|---|
| The data matrix | `X.shape` → `(150, 4)` |
| Grab a column | `X[:, 2]` |
| Filter by condition | `X[species == "Iris-setosa"]` |
| Math on everything at once | `petal_length * petal_width` |
| Summaries per column | `X.mean(axis=0)` |
| Standardize | `(X - X.mean(axis=0)) / X.std(axis=0)` |
| Find the extreme | `petal_area.argmax()` |

Now head to [exercises.md](exercises.md) — the last one has you build a tiny classifier by hand.